In [1]:
# Cell 1: Install (uncomment if needed) and imports
# If pygents & dependencies are already installed, you can skip the pip lines.

# !pip install pygents   # uncomment if not installed
# !pip install networkx matplotlib pandas scikit-learn

import os
import json
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report

# Add project path so this notebook can import your local pygents code if it's inside 'pygents' folder
cwd = os.getcwd()
if 'pygents' in os.listdir(cwd):
    project_path = cwd
else:
    # try parent directories (useful when notebook is inside repo subfolder)
    project_path = cwd
    for _ in range(4):
        if os.path.isdir(os.path.join(project_path, 'pygents')):
            break
        project_path = os.path.dirname(project_path)
if project_path not in os.sys.path:
    os.sys.path.append(project_path)

from pygents.aigents_api import PygentsSentiment, TextMetrics


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Cell 2: Load manual annotations (expected columns: 'sentence','positive','negative','neutral' or a 'label' column)
# Adjust the path if your CSV is elsewhere.

annotation_path = '/workspace/pygents/DialogRE/data/50_sentiment_annotation.csv'  # change if needed

df_gold = pd.read_csv(annotation_path)
print("Loaded annotations from", annotation_path)

display(df_gold.head())


Loaded annotations from /workspace/pygents/DialogRE/data/50_sentiment_annotation.csv


,subject,object,relation,conversation_id,sentence,psotive,negetive,neutral,strength
0,Speaker 2,Joey,per:positive_impression,conv_77,"[SPEAKER] Speaker 2: I don't know, you know, j...",1,0,0,0.9
1,Chandler,Speaker 2,per:parents,conv_252,"[SPEAKER] Speaker 1: Alright, so that leaves [...",0,0,1,0.0
2,Speaker 2,Frank,per:siblings,conv_5,"[SPEAKER] Speaker 2: Yeah, I did. I think it s...",0,1,0,-0.8
3,Janice,Speaker 1,per:girl/boyfriend,conv_68,[SPEAKER] Speaker 1: [E1]Janice[/E1]'s birthda...,1,0,0,1.0
4,Speaker 1,Frank Jr. Jr.,per:children,conv_149,[SPEAKER] Speaker 1: [E2]Frank Jr. Jr.[/E2]!!,0,0,1,0.0


In [3]:
# Cell 3: Initialize PygentsSentiment (paths below assume pygents/data/dict/en/* available)
# If your resource files are at different path change the strings accordingly.

p = PygentsSentiment('./data/dict/en/positive.txt',
                     './data/dict/en/negative.txt', debug=False)

def interpret_pygents_sentiment(sent_tuple):
    """
    Friendly wrapper that returns (label, pos_score, neg_score)
    - pos_score in [0..inf) (but typically small floats)
    - neg_score in [0..inf) (magnitude of negativity)
    The function is defensive if library returns different format.
    """
    if sent_tuple is None:
        return ('neutral', 0.0, 0.0)
    # Many versions of pygents: sentiment[1] == positive score, sentiment[2] == negative (negative number)
    if isinstance(sent_tuple, (list, tuple)):
        if len(sent_tuple) >= 3:
            pos = float(sent_tuple[1]) if sent_tuple[1] is not None else 0.0
            neg_raw = float(sent_tuple[2]) if sent_tuple[2] is not None else 0.0
            # if negative stored as negative, take absolute
            neg = abs(neg_raw)
            # decide label by comparing pos vs neg
            if pos > neg and pos > 0:
                lbl = 'positive'
            elif neg > pos and neg > 0:
                lbl = 'negative'
            else:
                lbl = 'neutral'
            return (lbl, pos, neg)
        # if a single scalar or different shape, try to coerce
        if len(sent_tuple) == 1:
            v = float(sent_tuple[0])
            if v > 0:
                return ('positive', v, 0.0)
            elif v < 0:
                return ('negative', 0.0, abs(v))
            else:
                return ('neutral', 0.0, 0.0)
    # final fallback
    return ('neutral', 0.0, 0.0)


FileNotFoundError: [Errno 2] No such file or directory: './data/dict/en/positive.txt'

In [10]:
# Cell 4: Run Pygents on gold sentences and compute per-class F1 for positive and negative
gold_labels = []
pred_labels = []
pos_scores = []
neg_scores = []

for _, row in df_gold.iterrows():
    sent = str(row.get('sentence',''))
    out = p.get_sentiment(sent)
    lbl, pos, neg = interpret_pygents_sentiment(out)
    pred_labels.append(lbl)
    pos_scores.append(pos)
    neg_scores.append(neg)

    # Decide gold label: prefer explicit 'positive'/'negative' columns if available; else try 'label' column
    if 'positive' in df_gold.columns and 'negative' in df_gold.columns:
        if int(row['positive']) == 1:
            gold = 'positive'
        elif int(row['negative']) == 1:
            gold = 'negative'
        else:
            gold = 'neutral'
    elif 'label' in df_gold.columns:
        gold = row['label']
    else:
        # fallback: assume neutral
        gold = 'neutral'
    gold_labels.append(gold)

results_df = df_gold.copy()
results_df['pred_label'] = pred_labels
results_df['pos_score'] = pos_scores
results_df['neg_score'] = neg_scores
results_df['gold_label'] = gold_labels

display(results_df)

# Compute binary F1 for positive: treat positive as 1, others as 0
y_true_pos = [1 if g=='positive' else 0 for g in gold_labels]
y_pred_pos = [1 if p=='positive' else 0 for p in pred_labels]

y_true_neg = [1 if g=='negative' else 0 for g in gold_labels]
y_pred_neg = [1 if p=='negative' else 0 for p in pred_labels]

from sklearn.metrics import f1_score, precision_score, recall_score

pos_precision = precision_score(y_true_pos, y_pred_pos, zero_division=0)
pos_recall = recall_score(y_true_pos, y_pred_pos, zero_division=0)
pos_f1 = f1_score(y_true_pos, y_pred_pos, zero_division=0)

neg_precision = precision_score(y_true_neg, y_pred_neg, zero_division=0)
neg_recall = recall_score(y_true_neg, y_pred_neg, zero_division=0)
neg_f1 = f1_score(y_true_neg, y_pred_neg, zero_division=0)

print("Positive -- precision: {:.3f}, recall: {:.3f}, f1: {:.3f}".format(pos_precision,pos_recall,pos_f1))
print("Negative -- precision: {:.3f}, recall: {:.3f}, f1: {:.3f}".format(neg_precision,neg_recall,neg_f1))

# Also show a small classification report
print("\nClassification report (macro):")
print(classification_report(gold_labels, pred_labels, zero_division=0))


,subject,object,relation,conversation_id,sentence,psotive,negetive,neutral,strength,pred_label,pos_score,neg_score,gold_label
0,Speaker 2,Joey,per:positive_impression,conv_77,"[SPEAKER] Speaker 2: I don't know, you know, j...",1,0,0,0.9,positive,0.48,0.43,neutral
1,Chandler,Speaker 2,per:parents,conv_252,"[SPEAKER] Speaker 1: Alright, so that leaves [...",0,0,1,0.0,positive,0.46,0.00,neutral
2,Speaker 2,Frank,per:siblings,conv_5,"[SPEAKER] Speaker 2: Yeah, I did. I think it s...",0,1,0,-0.8,positive,0.32,0.00,neutral
3,Janice,Speaker 1,per:girl/boyfriend,conv_68,[SPEAKER] Speaker 1: [E1]Janice[/E1]'s birthda...,1,0,0,1.0,positive,0.45,0.00,neutral
4,Speaker 1,Frank Jr. Jr.,per:children,conv_149,[SPEAKER] Speaker 1: [E2]Frank Jr. Jr.[/E2]!!,0,0,1,0.0,neutral,0.00,0.00,neutral
5,David,Speaker 3,per:girl/boyfriend,conv_224,"[SPEAKER] Speaker 3: [E1]David[/E1]'s like, y'...",1,0,0,0.4,positive,0.38,0.00,neutral
6,Monica,Speaker 1,per:siblings,conv_58,[SPEAKER] Speaker 1: You did! Oh.... I always ...,1,0,0,0.7,neutral,0.32,0.32,neutral
7,Barry,Speaker 1,per:girl/boyfriend,conv_251,"[SPEAKER] Speaker 1: Hey, you guys! Guess what...",0,1,0,-0.6,neutral,0.00,0.00,neutral
8,Speaker 2,Monica,per:spouse,conv_833,"[SPEAKER] Speaker 2: Oh uh, as it turns out, w...",0,1,0,-0.5,positive,0.35,0.00,neutral
9,Aunt Sheryl,Speaker 4,per:other_family,conv_52,"[SPEAKER] Speaker 4: Well l-look okay, it’s pr...",1,0,0,0.8,positive,0.47,0.40,neutral


Positive -- precision: 0.000, recall: 0.000, f1: 0.000
Negative -- precision: 0.000, recall: 0.000, f1: 0.000

Classification report (macro):
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         0
     neutral       1.00      0.34      0.51        50
    positive       0.00      0.00      0.00         0

    accuracy                           0.34        50
   macro avg       0.33      0.11      0.17        50
weighted avg       1.00      0.34      0.51        50



In [11]:
# Cell 5: Relation sentiment attribution
# Expected input df_rel columns: 'subject','object','sentence' (and optionally 'relation')
# We'll demonstrate with a small example; replace df_rel with your DialogRE relations dataframe.

# Example relation df (if you have your relation dataframe, set df_rel = your_dataframe)
if 'df_rel' not in globals():
    df_rel = pd.DataFrame([
        {"subject":"Bob","object":"Mary","sentence":"Bob likes Mary"},
        {"subject":"Mary","object":"Jane","sentence":"Mary hates Jane"},
        {"subject":"Jane","object":"Bob","sentence":"Jane adores and loves Bob very much."}
    ])

def assign_sentiment_to_relation(row, text_field='sentence'):
    # If you have a relation-specific span column use that instead
    text = row.get(text_field, '')
    out = p.get_sentiment(str(text))
    lbl, pos, neg = interpret_pygents_sentiment(out)
    # strength = choose the dominant polarity's absolute score; normalize (simple) to [0,1] by dividing by max observed or using tanh
    # We'll use a conservative normalization using logistic-ish scaling:
    raw_strength = pos if lbl=='positive' else (neg if lbl=='negative' else 0.0)
    # convert to [0,1] via simple saturating function
    strength = raw_strength / (raw_strength + 1.0) if raw_strength>0 else 0.0
    return pd.Series({"linkType":lbl, "strength":float(strength), "raw_pos":pos, "raw_neg":neg})

# apply to df_rel
df_rel_out = df_rel.join(df_rel.apply(assign_sentiment_to_relation, axis=1))
display(df_rel_out)


,subject,object,sentence,linkType,strength,raw_pos,raw_neg
0,Bob,Mary,Bob likes Mary,positive,0.435028,0.77,0.00
1,Mary,Jane,Mary hates Jane,negative,0.435028,0.00,0.77
2,Jane,Bob,Jane adores and loves Bob very much.,positive,0.363057,0.57,0.00


In [12]:
# Cell 7: End-to-end: starting from raw text -> relations -> sentiment links
# Here I assume you already have relation extraction (DialogRE) that yields triples.
# If you don't have a relation extraction component yet, you can use your DialogRE model inference code to produce df_rel.
# For demo, we'll convert your example input text and a hypothetical triple set:

input_text = "Bob likes Mary, Mary hates Jane, Jane adores and loves Bob very much."
# Hypothetical extracted relations (subject, object) for the example text:
df_example_rel = pd.DataFrame([
    {"subject":"Bob","object":"Mary","sentence":"Bob likes Mary"},
    {"subject":"Mary","object":"Jane","sentence":"Mary hates Jane"},
    {"subject":"Jane","object":"Bob","sentence":"Jane adores and loves Bob very much."}
])

# apply sentiment assignment
df_example_rel = df_example_rel.join(df_example_rel.apply(assign_sentiment_to_relation, axis=1))
display(df_example_rel)

# convert to links JSON (same as earlier)
links_example = []
for _, r in df_example_rel.iterrows():
    links_example.append({
        "source": r['subject'],
        "target": r['object'],
        "linkType": r['linkType'],
        "strength": float(r['strength'])
    })

print(json.dumps(links_example, indent=2))


,subject,object,sentence,linkType,strength,raw_pos,raw_neg
0,Bob,Mary,Bob likes Mary,positive,0.435028,0.77,0.00
1,Mary,Jane,Mary hates Jane,negative,0.435028,0.00,0.77
2,Jane,Bob,Jane adores and loves Bob very much.,positive,0.363057,0.57,0.00


[
  {
    "source": "Bob",
    "target": "Mary",
    "linkType": "positive",
    "strength": 0.4350282485875706
  },
  {
    "source": "Mary",
    "target": "Jane",
    "linkType": "negative",
    "strength": 0.4350282485875706
  },
  {
    "source": "Jane",
    "target": "Bob",
    "linkType": "positive",
    "strength": 0.36305732484076436
  }
]
